# Tips 09：ファイル名を一括変更

**業務の困りごと**：画像名に商品名と連番を付ける。

**このTipsでできること**：`enumerate(..., 1)` で1から始まる連番を作り、`Path.rename` でファイル名を変更します。既に同名のファイルがある場合は上書きしません。

**使用ライブラリ**：pathlib

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。各Tipsのサンプルはノートブック内で作ります。


## 1. 練習用データと準備

連番付きのファイル名を持つ練習用ファイルを3つ作ります。中身は名前変更の練習用ダミーです。

最後の `display()` で練習用データ全体を表として表示します。列名・行数・値を確認してから次へ進みましょう。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
from pathlib import Path
import tempfile
base=Path('/content') if Path('/content').exists() else Path.cwd()
# 毎回新しい練習用フォルダを作る
folder=Path(tempfile.mkdtemp(prefix='tips09_',dir=base))
# 名前変更用のダミーファイルを3件作る
for i in range(1, 4):
    (folder / f'IMG{i:03d}.jpg').write_bytes(b'practice-image-placeholder')
print([p.name for p in sorted(folder.glob('IMG*.jpg'))])
import pandas as pd
print('名前を変更するファイル（練習用データ）:')
display(pd.DataFrame([
    {'元ファイル名': p.name, '内容（名前変更用ダミー）': p.read_bytes().decode('utf-8')}
    for p in sorted(folder.glob('IMG*.jpg'))
]))


## 2. 自動処理

`enumerate(..., 1)` で1から始まる連番を作り、`Path.rename` でファイル名を変更します。既に同名のファイルがある場合は上書きしません。

**結果を見るポイント**：変更後に `商品A_001.jpg` から `商品A_003.jpg` ができ、ZIPでまとめて保存できます。

保存した成果物を読み直して `display()` で表示します。画面の表とダウンロードしたファイルの内容を照らし合わせてください。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
# 元のIMGファイルを名前順に取得する
files=sorted(folder.glob('IMG*.jpg'))
# 連番を付けて1件ずつ名前を変える
for i,path in enumerate(files,1):
    target=folder/f'商品A_{i:03d}.jpg'
    # 既存ファイルを誤って上書きしない
    if target.exists(): raise FileExistsError(target)
    path.rename(target)
print([p.name for p in sorted(folder.glob('商品A_*.jpg'))])
# 複数の成果物をPCでまとめて扱えるようZIPにする
import shutil
out = base / 'Tips09_成果物.zip'
shutil.make_archive(str(out.with_suffix('')), 'zip', root_dir=folder)
print('ダウンロード用ZIP:', out)

# 名前変更後のファイル一覧
display(pd.DataFrame([
    {'変更後のファイル名': p.name, '内容（名前変更用ダミー）': p.read_bytes().decode('utf-8')}
    for p in sorted(folder.glob('商品A_*.jpg'))
]))


## 3. 結果の確認

`assert` は条件が正しいか確認する命令です。条件が違えばそこで止まるため、想定外の集計を見つけられます。

変更後に `商品A_001.jpg` から `商品A_003.jpg` ができ、ZIPでまとめて保存できます。


In [ ]:
# 件数や合計が期待値と一致するか検査する
assert len(list(folder.glob('商品A_*.jpg')))==3
print('確認OK')
assert out.exists()  # ダウンロード用ZIPも確認する


## 4. 持ち帰り・練習

**練習問題**：接頭辞を別の商品名に変更してみましょう。

この演習のjpgは名前変更用のダミーです。実際の画像表示はできません。

**実務で使う前に**：練習用の `.jpg` は画像データではありません。実際の画像に置き換えれば同じ名前変更を試せます。

複数のファイルをZIPでPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
# Colab上で作った成果物をPCへダウンロードする
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
